# Setup

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json, glob, os, re
import numpy as np
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import TwoSlopeNorm, LogNorm
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
import os, sys, configparser

settings_parser = configparser.ConfigParser()
settings_parser.read("localsettings.ini")
MAIN_DATA_FOLDER = settings_parser.get("global", "data_path")
MIENC_PATH = settings_parser.get("global", "mienc_path")


band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
sns.set_theme("poster", "ticks")
sns.set_palette("colorblind")

cache_dir = os.path.join(MAIN_DATA_FOLDER, "cache")
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
FIGURES_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Figures")
if not os.path.isdir(FIGURES_FOLDER):
    os.mkdir(FIGURES_FOLDER)
if not os.path.isdir(os.path.join(RESULTS_FOLDER, "localised")):
    os.mkdir(os.path.join(RESULTS_FOLDER, "localised"))

sys.path.append(MIENC_PATH)
from mienc import Corrector

In [ ]:
def coloured_box_kwargs(colour):
    return {
        "boxprops": {"color": colour},
        "widths": 0.6,
        "whiskerprops": {"color": colour},
        "flierprops": {"markeredgecolor": colour, "marker": "+"},
        "medianprops": {"lw": 1.6, "color": colour},
        "capprops": {"color": colour},
    }

In [ ]:
def boxplot(
    x, series, colours, percents, ax=None, ttest=False, variant="Holm-Bonferroni"
):
    if ax is not None:
        plt.sca(ax)
    STRETCH = len(series)
    OFFSET = -STRETCH / 2
    handles = []
    labels = []

    for data, colour in zip(series, colours):
        plt.boxplot(
            series[data],
            positions=np.arange(len(x)) * STRETCH + OFFSET,
            **coloured_box_kwargs(colour),
        )
        handles.append(Patch(facecolor="white", edgecolor=colour))
        labels.append(data)
        OFFSET += 1

    if ttest:
        if hasattr(ttest, "__len__") and len(ttest) == 2:
            first, second = ttest
        elif len(series) == 2:
            first, second = series.keys()
        else:
            raise ValueError(ttest)
        significance = np.array(
            [
                ttest_rel(a, b, alternative="greater").pvalue
                for a, b in zip(series[first], series[second])
            ]
        )
        if variant == "Bonferroni":
            threshold = 0.05 / len(x)
        else:
            moving_threshold = 0.05 / (len(x) - np.arange(len(x)))
            significance_sorter = np.argsort(significance)
            above_threshold = significance[significance_sorter] > moving_threshold
            if above_threshold.any():
                first_above = np.min(np.where(above_threshold))
                threshold = significance[significance_sorter[first_above]]
            else:
                threshold = 0.05

        markers = significance < threshold
        position = ((max(percents) / 100 if percents else 0) + plt.ylim()[1]) / 2
        ast = plt.scatter(
            np.arange(len(x))[markers] * STRETCH - OFFSET / 2,
            np.full(np.sum(markers), position, dtype=float),
            marker="*",
            color="k",
            s=40,
        )
        handles.append(ast)
        labels.append("Significative\ndifference")

    for percent, style in zip(percents, ["solid", "dashed", "dashdot", "dotted"]):
        plt.hlines(
            percent / 100,
            -STRETCH,
            (len(x) - 0.5) * STRETCH,
            "r",
            style,
            lw=1,
            zorder=1,
        )
        handles.append(Line2D([0], [0], lw=1, color="r", linestyle=style))
        labels.append(f"{percent}%")

    tickPos = np.arange(len(x)) * STRETCH - 0.5
    plt.xticks(tickPos, x, rotation=90)
    plt.xlim((-0.8 * STRETCH, (len(x) - 0.6) * STRETCH))
    leg = plt.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        frameon=True,
        ncol=1,
    )

    t1 = leg.get_texts()
    # here we create the distinct instance
    for t in t1[1:]:
        t._fontproperties = t1[0]._fontproperties.copy()
    t1[len(series)].set_size(14)
    return plt.gca()

# Amount of non-linearity
## fMRI

In [ ]:
regions = [
    10,
    30,
    50,
    70,
    100,
    150,
    200,
    230,
    270,
    300,
    350,
    400,
    450,
    500,
    550,
    600,
    650,
    700,
    750,
    800,
    850,
    900,
    950,
]
fMRI_results = {"Empiric": [], "Shadow": []}
for region in regions:
    out_name = f"clean_cra_strin_{region}_bin7"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    fMRI_results["Empiric"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    fMRI_results["Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])

In [ ]:
fig = plt.figure(figsize=(12, 6))
boxplot(
    regions,
    fMRI_results,
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    plt.gca(),
    True,
    "Bonferroni",
)
plt.xlabel("# regions", labelpad=15)
plt.ylabel("RNL")
plt.ylim(plt.ylim())
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(regions) * 2 - 2, 4):
    plt.gca().add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
plt.savefig(os.path.join(FIGURES_FOLDER, "amount_fMRI.pdf"), bbox_inches="tight")
plt.show()

## EEG

In [ ]:
bands = ["delta", "theta", "alpha", "beta", "gamma"]
band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
nbins = [10, 13, 14, 20, 23]
EEG_results = {"Empiric": [], "Shadow": []}
for band, bin in zip(bands, nbins):
    out_name = f"EEG_124s_band_{band}_bin{bin}"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    EEG_results["Empiric"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    EEG_results["Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])

In [ ]:
fig = plt.figure(figsize=(12, 6))
boxplot(
    band_names,
    EEG_results,
    [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
    [0, 2, 5],
    plt.gca(),
    True,
    "Bonferroni",
)
plt.xlabel("Band", labelpad=15)
plt.ylabel("RNL")
plt.ylim(plt.ylim())
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(bands) * 2 - 2, 4):
    plt.gca().add_patch(
        Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
plt.savefig(os.path.join(FIGURES_FOLDER, "amount_EEG.pdf"), bbox_inches="tight")
plt.show()

## iEEG

In [ ]:
sessions = [1, "long"]
iEEG_results = {session: {"Empiric": [], "Shadow": []} for session in sessions}
for session in sessions:
    piece = session if session == "long" else f"ses{session:02}"
    band_folders = glob.glob(os.path.join(RESULTS_FOLDER, f"iEEG_{piece}_band*"))
    for folder in sorted(band_folders):
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        iEEG_results[session]["Empiric"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
        iEEG_results[session]["Shadow"].append(
            1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"]
        )

In [ ]:
for session in [1, "long"]:
    fig = plt.figure(figsize=(12, 6))
    boxplot(
        band_names,
        iEEG_results[session],
        [sns.color_palette("colorblind")[1], sns.color_palette("colorblind")[0]],
        [0, 1, 5],
        plt.gca(),
        True,
    )
    plt.xlabel("Band")
    plt.ylabel("RNL")
    plt.title(f"Session: {session}")
    plt.ylim(plt.ylim())
    bottom = plt.ylim()[0]
    height = plt.ylim()[1] - bottom
    for i in range(0, len(bands) * 2 - 2, 4):
        plt.gca().add_patch(
            Rectangle((i + 0.5, bottom), 2, height, fc=[0.85, 0.85, 0.85], zorder=0)
        )
    plt.savefig(
        os.path.join(
            FIGURES_FOLDER, f"amount_iEEG_{'short' if session==1 else session}.pdf"
        ),
        bbox_inches="tight",
    )
    plt.show()

## Single unit spikes

In [ ]:
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]
groups = {}
statsReg = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(
        os.path.join(RESULTS_FOLDER, f"spiking_{session}_????_bin*")
    ):
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            with open(os.path.join(folder, "shape.json")) as fp:
                groups[session] = json.load(fp)[1]
            timeWindow = int(re.findall(r"_(\d{4})_", folder)[0])
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            statsReg[s][timeWindow] = {
                "Empiric": 1 - tmp["gaussMI"] / tmp["totalMI"],
                "Shadow": 1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"],
            }

In [ ]:
for data in ["Empiric", "Shadow"]:
    spikes = np.empty((len(statsReg[0]), len(statsReg[0][125][data]), len(statsReg)))
    ylabels = list(sorted(statsReg[0].keys()))
    xlabels = [2**i for i in range(6)]
    for s in range(len(GOOD_SESSIONS)):
        for t, timeWindow in enumerate(sorted(statsReg[0].keys())):
            spikes[t, :, s] = statsReg[s][timeWindow][data]

    plt.imshow(np.mean(spikes, 2))
    plt.yticks(np.arange(7), ylabels)
    plt.xticks(np.arange(6), xlabels)
    cbar = plt.colorbar()
    cbar.ax.set_ylabel("RNL", rotation=90)
    cbar.ax.get_yaxis().labelpad = 15
    plt.xlabel("# units per site")
    plt.ylabel("Time sample [ms]")
    plt.title(f"Average $-$ {data}", pad=15)
    plt.savefig(
        os.path.join(FIGURES_FOLDER, f"amount_SUS_{data}.pdf"), bbox_inches="tight"
    )
    plt.show()

# Localisation of non-linearities


In [ ]:
from s03a_localisation import (
    compute_localised_non_linearity,
    show_localised_non_linearity,
)

## fMRI

In [ ]:
pair_num = 4005
subj_num = 245
regions_num = 90
cut_pos = (-15, -75, 27)
results_file = os.path.join(
    RESULTS_FOLDER, "eso245_aal_{0}_90_bin{1}/subject{2:02}_{1}.npy"
)
subset_description = "fMRI AAL90 "
subset_identifiers = ["raw", "mod", "strin"]  # ["raw"]#
subset_names = ["Raw", "Moderate", "Stringent"]  # ["strin"]#
output_prefix = "fMRI_AAL90"
nbins = 7
with open(
    os.path.join(os.path.dirname(results_file.format("strin", nbins, 0)), "shape.json")
) as fp:
    samples = json.load(fp)[0]
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    samples,
)
results_file = os.path.join(
    RESULTS_FOLDER, "eso245_aal_{0}_90_bin{1}/subject{2:02}_{1}_sha.npy"
)
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix + "_sha",
    pair_num,
    subj_num,
    nbins,
    samples,
)
show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    regions_num,
    FIGURES_FOLDER,
    cut_pos,
)

## EEG

In [ ]:
pair_num = 18915
subj_num = 50
elec_num = 195
results_file = os.path.join(
    RESULTS_FOLDER, "EEG_124s_band_{0}_bin{1}/subject{2:02}_{1}.npy"
)
subset_description = "EEG "
subset_identifiers = ["delta", "theta", "alpha", "beta", "gamma"]
subset_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
output_prefix = "EEG_195"
nbins = [10, 13, 14, 20, 23]
samples = []
for band, bins in zip(subset_identifiers, nbins):
    with open(
        os.path.join(os.path.dirname(results_file.format(band, bins, 0)), "shape.json")
    ) as fp:
        samples.append(json.load(fp)[0])

compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix,
    pair_num,
    subj_num,
    nbins,
    samples,
)

results_file = os.path.join(
    RESULTS_FOLDER, "EEG_124s_band_{0}_bin{1}/subject{2:02}_{1}_sha.npy"
)
compute_localised_non_linearity(
    results_file,
    subset_identifiers,
    output_prefix + "_sha",
    pair_num,
    subj_num,
    nbins,
    samples,
)

show_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    elec_num,
    FIGURES_FOLDER,
)

# Sources of non-linearity

## EEG

In [ ]:
bands = ["delta", "theta", "alpha", "beta", "gamma"]
band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
BLP_results = {"Empiric": [], "Naive\nShadow": [], "Shadow": []}
for band in bands:
    out_name = f"electrodeBLP_band_{band}_bin10"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    BLP_results["Empiric"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    BLP_results["Naive\nShadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])
    out_name = f"electrodeBLP_shadow_band_{band}_bin10"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    BLP_results["Shadow"].append(1 - tmp["gaussMI"] / tmp["totalMI"])

In [ ]:
fig = plt.figure(figsize=(12, 6))
boxplot(
    band_names,
    BLP_results,
    [
        sns.color_palette("colorblind")[1],
        sns.color_palette("colorblind")[0],
        sns.color_palette("colorblind")[2],
    ],
    [0, 2, 5],
    plt.gca(),
    ["Empiric", "Shadow"],
    "Bonferroni",
)
plt.xlabel("Band", labelpad=15)
plt.ylabel("RNL")
plt.ylim(plt.ylim())
bottom = plt.ylim()[0]
height = plt.ylim()[1] - bottom
for i in range(0, len(bands) * 4 - 8, 6):
    plt.gca().add_patch(
        Rectangle((i + 1, bottom), 3, height, fc=[0.85, 0.85, 0.85], zorder=0)
    )
plt.savefig(os.path.join(FIGURES_FOLDER, "sources_EEG.pdf"), bbox_inches="tight")
plt.show()

## iEEG

### Seizure

In [ ]:
from scipy import signal

fs_fast = 200
bands = [[1.0, 4.0], [4.0, 8.0], [8.0, 12.0], [12.0, 30.0], [30.0, 44.0]]

seizure = {}
for folder in glob.glob(os.path.join(RESULTS_FOLDER, "sampleSeizure_band?_bin*")):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        band = int(re.findall(r"band(\d+)", folder)[0])
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        seizure[band] = {
            "Empiric": (tmp["totalMI"] - tmp["gaussMI"]) / tmp["totalMI"],
            "Shadow": (tmp["totalMIshadow"] - tmp["gaussMIshadow"])
            / tmp["totalMIshadow"],
        }
windowed_mean_power = np.load(
    os.path.join(MAIN_DATA_FOLDER, "iEEG", "sampleSeizure_power.npy")
)
original_iEEG = loadmat(os.path.join(MAIN_DATA_FOLDER, "iEEG", "sampleSeizure.mat"))[
    "EEG"
]
start = 1782743 - 51200
stop = 1971102 + 102400
resam, t = signal.resample(
    original_iEEG[4::8, start:stop].T,
    round((stop - start) / 1024 * fs_fast),
    np.arange(stop - start) / 1024,
)

In [ ]:
RNL = []

sec_dest = 283.936
for i in range(1, 6):
    thisRNL = seizure[i]["Empiric"]
    N_blocks = thisRNL.shape[0]
    fs_dest = int(bands[i - 1][1] * 2 * 1.125 + 0.5)
    slide_band = (fs_dest * 45) // 10
    fs_rnl = slide_band / fs_dest
    thisTime = fs_rnl * N_blocks
    tot_samp_dest = thisTime * fs_fast
    conversion_factor = tot_samp_dest / (fs_fast * fs_rnl)
    index = np.round(np.arange(tot_samp_dest) / (fs_fast * fs_rnl)).astype(int)
    index[index == thisRNL.shape[0]] = thisRNL.shape[0] - 1
    RNL.append(thisRNL[index])
tmpRNL = np.concatenate(RNL)
normalised_wmp = (windowed_mean_power - np.min(windowed_mean_power)) / (
    np.max(windowed_mean_power) - np.min(windowed_mean_power)
) * (np.nanmax(tmpRNL) - np.nanmin(tmpRNL)) + np.nanmin(tmpRNL)
N_blocks = 64
fs_dest = 1024
slide_band = (fs_dest * 45) // 10
fs_rnl = slide_band / fs_dest
thisTime = fs_rnl * N_blocks
tot_samp_dest = thisTime * fs_fast
conversion_factor = tot_samp_dest / (fs_fast * fs_rnl)
index = np.round(np.arange(tot_samp_dest) / (fs_fast * fs_rnl)).astype(int)
index[index == normalised_wmp.shape[0]] = normalised_wmp.shape[0] - 1
RNL.append(normalised_wmp[index])

In [ ]:
tot_samp = max(map(lambda x: x.shape[0], RNL))
RNL_long = np.full([tot_samp, 6], np.nan)
for b in range(6):
    RNL_long[: RNL[b].shape[0], b] = RNL[b]

In [ ]:
plt.figure(figsize=(15, 8))
plt.imshow(RNL_long.T, interpolation="nearest")
plt.gca().set_aspect(5000)
# plt.show()
plt.xlabel("Time [s]")
plt.ylabel("Band                  Electrode      ")
plt.xticks(
    np.arange(12) * 5000,
    (np.arange(12) * 5000 / fs_fast).astype(int),
    rotation=30,
    ha="right",
    rotation_mode="anchor",
)
plt.yticks(
    np.arange(6),
    ["power\n(a.u.)", r"$\gamma$", r"$\beta$", r"$\alpha$", r"$\theta$", r"$\delta$"][
        ::-1
    ],
)
# plt.ylim(plt.ylim())
plt.plot(t * 200, resam / 2000 + np.arange(7) + 6, "k", lw=0.5, alpha=0.5)
plt.ylim((-0.5, 13))
plt.vlines(
    [50 * fs_fast, (183.936 + 50) * fs_fast],
    *plt.ylim(),
    colors=sns.color_palette("colorblind")[2],
    linestyles="solid",
    lw=2,
    zorder=20
)
plt.vlines(
    [(50 - 45) * fs_fast, (183.936 + 50 - 45) * fs_fast],
    -0.5,
    5.5,
    colors=sns.color_palette("colorblind")[0],
    linestyles="solid",
    lw=2,
    zorder=20,
)
plt.colorbar(shrink=0.7).ax.set_ylabel("RNL", rotation=270, labelpad=15)
plt.xlim((0, 283.936 * fs_fast))
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sources_iEEG_seizure.pdf"), bbox_inches="tight"
)
plt.show()

### Other Activity

In [ ]:
with open(
    os.path.join(MAIN_DATA_FOLDER, "iEEG", "spikes_by_subject_electrode.json")
) as fp:
    spikes_per_electrode = {
        int(k): np.array([len(v[a][0]) for a in v]) for k, v in json.load(fp).items()
    }
bands = [[1.0, 4.0], [4.0, 8.0], [8.0, 12.0], [12.0, 30.0], [30.0, 44.0]]
fs_dest = [int(band[1] * 2 * 1.125 + 0.5) for band in bands]

In [ ]:
fig = plt.figure(figsize=(8, 6))
spikes_per_subject = [
    np.sum(spikes_per_electrode[k]) for k in sorted(spikes_per_electrode.keys())
]
for band, (rnl, fmt) in enumerate(zip(iEEG_results[1]["Empiric"], "+^s3o"), start=1):
    r = pearsonr(spikes_per_subject, rnl)
    label = (
        f"{band_names[band - 1]} - r:{r.statistic:.3} {'*' if r.pvalue<0.01 else ''}"
    )
    p = plt.scatter(spikes_per_subject, rnl, marker=fmt, label=label)
    par = np.polyfit(spikes_per_subject, rnl, 1)
    plt.xlim(plt.xlim())
    plt.plot(
        plt.xlim(),
        np.polyval(par, plt.xlim()),
        color=p.get_facecolor(),
        lw=2 if r.pvalue < 0.01 else 0.8,
        ls="-" if r.pvalue < 0.01 else ":",
    )
plt.xlabel("#IED")
plt.ylabel("RNL")
plt.legend(loc="center left", title="Band", bbox_to_anchor=(1, 0.5), frameon=False)
plt.savefig(os.path.join(FIGURES_FOLDER, "sources_iEEG_IED.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
def data_from(band, nbins, duration):
    MI_thres = {10: 0.015696, 13: 0.009054, 14: 0.008483, 20: 0.003671, 23: 0.002931}
    corrct = Corrector(
        nbins,
        duration,
        folder_name=os.path.join(RESULTS_FOLDER, f"iEEG_ses01_band{band}_bin{nbins}"),
        config="../config.ini",
        cache_dir="../cache/",
        display=False,
        verbose=True,
    )
    corrct.compute_correction()
    ns = []
    rnl = []
    aga = []
    for subj in range(18):
        MI = corrct.correct(
            np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"iEEG_ses01_band{band}_bin{nbins}",
                    f"subject{subj:02}_{nbins}.npy",
                )
            )
        )
        tr = MI[:, 0]
        ga = np.mean(MI[:, 1:], axis=1)
        rnl.append(1 - ga / tr)
        aga.append(ga)
        fe, se = np.triu_indices(24, 1)
        ns.append(
            spikes_per_electrode[subj + 1][fe]
            + spikes_per_electrode[subj + 1][se]  # [band-1][band-1]
        )
    ns = np.concatenate(ns)
    aga = np.concatenate(aga)
    rnl = np.concatenate(rnl)
    good = (rnl >= 0) & np.isfinite(rnl) & (aga > MI_thres[nbins])
    return ns, rnl, good

In [ ]:
data = [
    data_from(band, nbins, duration)
    for band, (duration, nbins) in enumerate(  # 23, 12276
        [(1116, 10), (2232, 13), (3348, 14), (8432, 20), (12276, 23)], start=1
    )
]
plt.close()
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=None, sharey="all")
plt.subplots_adjust(hspace=0.13, wspace=0)
for band, (ns, rnl, good) in enumerate(data, start=1):
    print(np.sum(rnl > 1))
    good = np.isfinite(rnl) & (rnl > -5)
    his = ax[(band - 1) // 3, (band - 1) % 3].hist2d(
        ns[good],
        rnl[good],
        norm=LogNorm(),
        bins=[np.arange(11) * 25, np.arange(13) / 2 - 5],
    )
    r = pearsonr(ns[good], rnl[good])
    ax[(band - 1) // 3, (band - 1) % 3].set_title(
        f"{band_names[band - 1]} - r:{r.statistic:.3} {'*' if r.pvalue<0.01 else ''}",
        y=1.0,
        pad=5,
    )
    if (band - 1) // 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_xlabel("# IED")
    if not (band - 1) % 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_ylabel("RNL")
plt.colorbar(his[-1], ax=ax[1, 2]).set_label("# pairs")
ax[1, 2].axis("off")
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sources_iEEG_IED_all.pdf"), bbox_inches="tight"
)
plt.show()
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=None, sharey="all")
plt.subplots_adjust(hspace=0.13, wspace=0)
for band, (ns, rnl, good) in enumerate(data, start=1):
    his = ax[(band - 1) // 3, (band - 1) % 3].hist2d(
        ns[good], rnl[good], norm=LogNorm()
    )
    r = pearsonr(ns[good], rnl[good])
    ax[(band - 1) // 3, (band - 1) % 3].set_title(
        f"{band_names[band - 1]} - r:{r.statistic:.3} {'*' if r.pvalue<0.01 else ''}",
        y=1.0,
        pad=5,
    )
    if (band - 1) // 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_xlabel("# IED")
    if not (band - 1) % 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_ylabel("RNL")
plt.colorbar(his[-1], ax=ax[1, 2]).set_label("# pairs")
ax[1, 2].axis("off")
plt.ylim(bottom=0, top=1)
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sources_iEEG_IED_significantPairs.pdf"),
    bbox_inches="tight",
)
plt.show()

## Single unit spikes
### Time evolution

In [ ]:
import matplotlib.colors as mplcol

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

SAMPLE_SESSION = 779839471  # 794812542#778998620

TL = 5  # 32 units
r1 = 5
r2 = 9
filename = os.path.join(MAIN_DATA_FOLDER, "SUS", f"spiking_{SAMPLE_SESSION}_1000.mat")
mat = loadmat(filename)["spikes"][:, :, TL]

name = filename.split("_")[2]
el_num = mat.shape[1]
el_pair = int(el_num * (el_num - 1) / 2)

corr1 = np.empty((30, el_pair))
for i in np.arange(30):
    corr1[i, :] = np.corrcoef(mat[60 * i : 60 * (i + 1), :], rowvar=False)[
        np.triu_indices(el_num, 1)
    ]

corr2 = np.empty((30, 30))
corr2[:, :] = np.corrcoef(corr1[:, :], rowvar=True)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plt.sca(ax)
plt.imshow(corr2[:, :])
plt.colorbar(shrink=0.8)
plt.title(2**TL)
plt.xticks([0, 5, 10, 15, 20, 25])
plt.yticks([])
plt.suptitle(name)
plt.subplots_adjust(wspace=0, hspace=0)
plt.savefig(os.path.join(FIGURES_FOLDER, "sources_SUS_matrix.pdf"), bbox_inches="tight")
plt.show()


filename = os.path.join(MAIN_DATA_FOLDER, "SUS", f"spiking_{SAMPLE_SESSION}_2000.mat")
mat = loadmat(filename)["spikes"][:, :, TL]
plt.scatter(
    mat[:, r1],
    mat[:, r2],
    c=np.arange(mat.shape[0]) / (mat.shape[0] - 1),
    cmap=plt.cm.jet,
    alpha=0.8,
    s=15,
)  # plt.cm.Set3, alpha=0.7)
cbar = plt.colorbar()
cbar.ax.set_ylabel("Time [minutes]", rotation=90)
cbar.ax.set_yticks([0, 1 / 3, 2 / 3, 1])
cbar.ax.set_yticklabels([0, 10, 20, 30])
cbar.ax.get_yaxis().labelpad = 15
plt.savefig(
    os.path.join(FIGURES_FOLDER, "sources_SUS_timeScatter.pdf"), bbox_inches="tight"
)
plt.show()

### Epochs

In [ ]:
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]
groups = {}
statsRegSpl = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(os.path.join(RESULTS_FOLDER, f"spiking_{session}_0[125]*")):
        if "spl" in folder:
            matched = re.findall(f"_spl(\d+)_part(\d+)_", folder)[0]
            spl = int(matched[0])
            part = int(matched[1])
        else:
            spl = 1
            part = 0
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            with open(os.path.join(folder, "shape.json")) as fp:
                groups[session] = json.load(fp)[1]
            timeWindow = int(re.findall(r"_(\d{4})_", folder)[0])
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            statsRegSpl[s][(timeWindow, spl, part)] = {
                "Empiric": 1 - tmp["gaussMI"] / tmp["totalMI"],
                "Shadow": 1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"],
            }

In [ ]:
avgRNL = np.zeros((6, 3, 6))
for s, spl in enumerate([1, 2, 3, 5, 6, 10]):
    N = 0
    for stat in statsRegSpl:
        tmp = np.zeros((3, 6))
        try:
            for t, time in enumerate([125, 250, 500]):
                for p in range(spl):
                    tmp[t, :] += stat[(time, spl, p)]["Empiric"]
        except:
            continue
        N += 1
        avgRNL[s, :, :] += tmp
    avgRNL[s, :, :] /= N * spl

In [ ]:
vmax = np.max(avgRNL)
vmin = np.min(avgRNL)
for s, spl in enumerate([1, 2, 3, 5, 6, 10]):
    plt.imshow(avgRNL[s, :, :], vmax=vmax, vmin=vmin)
    cbar = plt.colorbar(shrink=0.6)
    cbar.ax.set_ylabel("RNL", rotation=90)
    cbar.ax.get_yaxis().labelpad = 15

    ylabels = [125, 250, 500]
    xlabels = [2**i for i in range(6)]
    plt.yticks(np.arange(3), ylabels)
    plt.xticks(np.arange(6), xlabels)
    plt.xlabel("# units per site")
    plt.ylabel("Time sample [ms]")
    plt.title(f"Epoch duration: {30//spl} minutes", pad=15)
    plt.savefig(
        os.path.join(FIGURES_FOLDER, f"sources_SUS_epochs{spl}.pdf"),
        bbox_inches="tight",
    )
    plt.show()

# Reliability of non-linearity
## fMRI test-retest

In [ ]:
corrector = Corrector(
    bins=8,
    duration=657,
    steps=200,
    iterations=50000,
    samples=657,
    folder_name=os.path.join(RESULTS_FOLDER, "lemon_aal_strin_90_ses-01_bin8"),
    cache_dir=cache_dir,
    verbose=True,
)
corrector.compute_correction()
all_sub = np.empty([3, 6, 14])
all_cov = np.empty([6, 6, 14])
for subject in range(14):
    all_series = np.empty([6, 4005])
    for session in range(3):
        all_series[2 * session, :] = corrector.correct(
            np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8.npy",
                )
            )[:, 0]
        )
        all_series[2 * session + 1, :] = -0.5 * np.log10(
            1
            - np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8_cor.npy",
                )
            )
            ** 2
        )
    cov = np.corrcoef(all_series)
    np.fill_diagonal(cov, np.nan)
    all_cov[:, :, subject] = cov
    all_sub[:, :, subject] = np.diff(cov, axis=0)[::2, :]

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.mean(all_cov, axis=2))
plt.colorbar()
plt.yticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
)
plt.gca().set_facecolor((0.85, 0.85, 0.85))
plt.title("Average across 14 subjects", pad=15)
plt.savefig(
    os.path.join(FIGURES_FOLDER, f"reliability_fMRI_matrix.pdf"), bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(np.mean(all_sub, axis=-1))
plt.colorbar(shrink=0.7).ax.set_ylabel(
    r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$",
    fontsize=14,
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)], fontsize=15)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
    fontsize=15,
)
plt.gca().set_facecolor((0.85, 0.85, 0.85))
plt.ylabel("Predictors from:", fontsize=16)
plt.xlabel("Predicting:", fontsize=16)
plt.title("Average across 14 subjects", fontsize=18)
plt.savefig(
    os.path.join(FIGURES_FOLDER, f"reliability_fMRI_improvement.pdf"),
    bbox_inches="tight",
)
plt.show()

In [ ]:
corrector = Corrector(
    bins=8,
    duration=657,
    steps=200,
    iterations=50000,
    samples=657,
    folder_name=os.path.join(RESULTS_FOLDER, "lemon_aal_strin_90_ses-01_bin8"),
    cache_dir=cache_dir,
    verbose=True,
)
corrector.compute_correction()
all_cov = np.empty([14, 6, 4005])
for subject in range(14):
    for session in range(3):
        all_cov[subject, 2 * session, :] = corrector.correct(
            np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8.npy",
                )
            )[:, 0]
        )
        all_cov[subject, 2 * session + 1, :] = -0.5 * np.log10(
            1
            - np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8_cor.npy",
                )
            )
            ** 2
        )

In [ ]:
pecorr = np.empty([6, 6, 4005])
for c in range(4005):
    pecorr[:, :, c] = np.corrcoef(
        np.argsort(np.argsort(all_cov[:, :, c], axis=0), axis=0), rowvar=False
    )
cov = np.mean(pecorr, axis=2)
np.fill_diagonal(cov, np.nan)

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(cov)
plt.colorbar()
plt.yticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
)
plt.title("Average across 4005 pairs", pad=15)
plt.gca().set_facecolor((0.85, 0.85, 0.85))
plt.savefig(
    os.path.join(FIGURES_FOLDER, f"reliability_fMRI_matrix_subjectRank.pdf"),
    bbox_inches="tight",
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.imshow(np.diff(cov, axis=0)[::2, :])
plt.colorbar(shrink=0.7).ax.set_ylabel(
    "Correlation improvement\nusing Gaussian estimate", fontsize=16
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])  # , fontsize=15)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=20,
    ha="right",
    rotation_mode="anchor",
    # fontsize=15,
)
plt.ylabel("Predictors from:")  # , fontsize=16)
plt.xlabel("Predicting:")  # , fontsize=16)
plt.title("Average across 4005 pairs", pad=15)  # , fontsize=18)
plt.gca().set_facecolor((0.85, 0.85, 0.85))
plt.savefig(
    os.path.join(FIGURES_FOLDER, f"reliability_fMRI_improvement_subjectRank.pdf"),
    bbox_inches="tight",
)
plt.show()

## EEG

## iEEG

In [ ]:
all_sub = {band: np.empty([3, 6, 18]) for band in range(1, 6)}
all_cov = {band: np.empty([6, 6, 18]) for band in range(1, 6)}
all_spe = {band: np.empty([18, 6, 276]) for band in range(1, 6)}
for band in range(1, 6):
    for subject in range(18):
        data = {}
        for session in range(3):
            band_folder = glob.glob(
                os.path.join(RESULTS_FOLDER, f"iEEG_ses{session+1:02}_band{band}_bin*")
            )[0]
            bins = int(band_folder.split("bin")[1])
            with open(os.path.join(band_folder, "shape.json"), "r") as fp:
                duration = json.load(fp)[0]
            corrector = Corrector(
                bins=bins,
                duration=duration,
                steps=200,
                iterations=50000,
                samples=duration,
                folder_name=band_folder,
                cache_dir=cache_dir,
                verbose=False,
            )
            corrector.compute_correction()
            all_spe[band][subject, 2 * session, :] = corrector.correct(
                np.load(os.path.join(band_folder, f"subject{subject:02}_{bins}.npy"))[
                    :, 0
                ]
            )
            all_spe[band][subject, 2 * session + 1, :] = -0.5 * np.log10(
                1
                - np.load(
                    os.path.join(band_folder, f"subject{subject:02}_{bins}_cor.npy")
                )
                ** 2
            )
        cov = np.corrcoef(all_spe[band][subject, :, :])
        np.fill_diagonal(cov, np.nan)
        all_cov[band][:, :, subject] = cov
        all_sub[band][:, :, subject] = np.diff(cov, axis=0)[::2, :]

In [ ]:
for band in all_cov:
    plt.figure(figsize=(10, 10))
    plt.imshow(np.mean(all_cov[band], axis=2))
    plt.colorbar()
    plt.yticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    )
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=90,
    )
    plt.title(f"Average across 18 subjects $-$ Band {band_names[band-1]}", pad=15)
    plt.gca().set_facecolor((0.85, 0.85, 0.85))
    plt.savefig(
        os.path.join(FIGURES_FOLDER, f"reliability_iEEG_matrix_band{band}.pdf"),
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
band_success = {"TMI": [], "Cor": []}
allontmi = []
alloncor = []
for band in all_sub:
    mean_all_sub = np.mean(all_sub[band], axis=-1)
    max_abs = np.nanmax(np.abs(mean_all_sub))
    norm = TwoSlopeNorm(vcenter=0, vmax=max_abs, vmin=-max_abs)
    plt.figure(figsize=(15, 10))
    plt.imshow(mean_all_sub, cmap=plt.cm.RdYlBu_r, norm=norm)
    plt.colorbar(shrink=0.7).ax.set_ylabel(
        r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$",
        fontsize=14,
    )
    plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)], fontsize=15)
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=90,
        fontsize=15,
    )
    plt.ylabel("Predictors from:", fontsize=16)
    plt.xlabel("Predicting:", fontsize=16)
    plt.title(f"Average across 18 subjects $-$ Band {band_names[band-1]}", fontsize=18)
    plt.savefig(
        os.path.join(FIGURES_FOLDER, f"reliability_iEEG_improvement_band{band}.pdf"),
        bbox_inches="tight",
    )
    plt.gca().set_facecolor((0.85, 0.85, 0.85))
    plt.show()
    band_success["TMI"].append(np.nanmean(mean_all_sub[:, ::2]))
    band_success["Cor"].append(np.nanmean(mean_all_sub[:, 1::2]))
    allontmi.append(mean_all_sub[:, ::2].flatten())
    alloncor.append(mean_all_sub[:, 1::2].flatten())
    # allontmi.append(all_sub[band][:,::2,:].flatten())
    # alloncor.append(all_sub[band][:,1::2,:].flatten())
for k in band_success:
    print(f"{k}: {np.mean(band_success[k]):.3}")

In [ ]:
from scipy.stats import ttest_1samp

allontmilong = np.concatenate(allontmi)
print(
    "TMI",
    ttest_1samp(allontmilong[np.isfinite(allontmilong)], 0, alternative="two-sided"),
)  # "less"))
alloncorlong = np.concatenate(alloncor)
print(
    "cor",
    ttest_1samp(alloncorlong[np.isfinite(alloncorlong)], 0, alternative="two-sided"),
)  # "greater"))

In [ ]:
pecorr = {band: np.empty([6, 6, 276]) for band in range(1, 6)}
for band in range(1, 6):
    for c in range(276):
        pecorr[band][:, :, c] = np.corrcoef(
            np.argsort(np.argsort(all_spe[band][:, :, c], axis=0), axis=0), rowvar=False
        )
cov = {band: np.mean(pecorr[band], axis=2) for band in range(1, 6)}
for band in range(1, 6):
    np.fill_diagonal(cov[band], np.nan)

In [ ]:
for band in range(1, 6):
    plt.figure(figsize=(10, 10))
    plt.imshow(cov[band])
    plt.colorbar()
    plt.yticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    )
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=90,
    )
    plt.title(f"Average across 276 pairs $-$ Band {band_names[band-1]}", pad=15)
    plt.gca().set_facecolor((0.85, 0.85, 0.85))
    plt.savefig(
        os.path.join(
            FIGURES_FOLDER, f"reliability_iEEG_matrix_subjectRank_band{band}.pdf"
        ),
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
for band in range(1, 6):
    plt.figure(figsize=(10, 7))
    mean_all_pair = np.diff(cov[band], axis=0)[::2, :]
    max_abs = np.nanmax(np.abs(mean_all_pair))
    norm = TwoSlopeNorm(vcenter=0, vmax=max_abs, vmin=-max_abs)
    plt.imshow(mean_all_pair, cmap=plt.cm.RdYlBu_r, norm=norm)
    plt.colorbar(shrink=0.7).ax.set_ylabel(
        "Correlation improvement\nusing Gaussian estimate", fontsize=16
    )
    plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)])  # , fontsize=15)
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=20,
        ha="right",
        rotation_mode="anchor",
        # fontsize=15,
    )
    plt.ylabel("Predictors from:")  # , fontsize=16)
    plt.xlabel("Predicting:")  # , fontsize=16)
    plt.title(
        f"Average across  276 pairs $-$ Band {band_names[band-1]}", pad=15
    )  # , fontsize=18)
    plt.gca().set_facecolor((0.85, 0.85, 0.85))
    plt.savefig(
        os.path.join(
            FIGURES_FOLDER, f"reliability_iEEG_improvement_subjectRank_band{band}.pdf"
        ),
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
from mienc.support import get_pool, task_producer

with get_pool(1) as pool:
    for i in pool.map(int, np.arange(10) / 2):
        print(i)

In [ ]:
random_state = np.random.default_rng(42)
print(random_state.bit_generator.__getstate__())
pat = random_state.normal(0, 0.5, [100, 50])
print(random_state.bit_generator.__getstate__())
for i in task_producer(pat, 5, random_state=random_state):
    print(random_state.bit_generator.__getstate__())